# Similarity and next-word probabilities

This notebook goes with the Detailed lesson **Natural Language Processing: Tokens, Embeddings and Language Models**, section 5.

We do vector arithmetic on a small word map, measure cosine similarity, and build a bigram language model. Run each cell in order with **Shift + Enter**.

In [ ]:
import numpy as np
from collections import Counter, defaultdict

words = {
    "king": (8.5, 8.5), "queen": (8.5, 6.5), "man": (5.0, 8.5), "woman": (5.0, 6.5),
    "dog": (2.5, 8.5), "puppy": (2.5, 6.5), "cat": (1.5, 8.5), "kitten": (1.5, 6.5),
    "france": (8.0, 2.5), "paris": (8.0, 4.5), "japan": (3.5, 2.5), "tokyo": (3.5, 4.5),
}
vec = {w: np.array(p) for w, p in words.items()}

## 1. Analogies as arithmetic

Compute a − b + c, then find the closest word that is not one of the three inputs.

In [ ]:
def analogy(a, b, c):
    target = vec[a] - vec[b] + vec[c]
    candidates = [w for w in vec if w not in (a, b, c)]
    best = min(candidates, key=lambda w: np.linalg.norm(vec[w] - target))
    print(f"{a} - {b} + {c} = {target} -> closest word: {best}")

analogy("king", "man", "woman")
analogy("paris", "france", "japan")
analogy("puppy", "dog", "cat")

## 2. Cosine similarity

Dot product divided by the product of the lengths: 1 means the same direction, 0 means unrelated.

In [ ]:
def cosine(a, b):
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("(3, 4) and (4, 3): ", round(cosine((3, 4), (4, 3)), 2))
print("(3, 4) and (4, -3):", round(cosine((3, 4), (4, -3)), 2))
print("king and queen:    ", round(cosine(vec["king"], vec["queen"]), 2))

## 3. A bigram language model

Count which word follows which, then turn the counts into probabilities.

In [ ]:
corpus = ["the cat sat", "the cat ran", "the dog sat"]

follows = defaultdict(Counter)
for sentence in corpus:
    tokens = sentence.split()
    for first, second in zip(tokens, tokens[1:]):
        follows[first][second] += 1

def p(next_word, word):
    total = sum(follows[word].values())
    return follows[word][next_word] / total if total else 0.0

for word in ["the", "cat", "dog"]:
    print(word, "->", {w: f"{c}/{sum(follows[word].values())}" for w, c in follows[word].items()})

In [ ]:
def sentence_probability(sentence):
    tokens = sentence.split()
    prob = 1.0
    for first, second in zip(tokens, tokens[1:]):
        prob *= p(second, first)
    return prob

for s in ["the cat sat", "the dog sat", "the dog ran"]:
    print(f"P({s!r}) = {sentence_probability(s):.3f}")

## 4. Generating text

Repeatedly pick the most likely next word. This is the same loop a large language model runs, with a far better model.

In [ ]:
def generate(word, length=3):
    out = [word]
    for _ in range(length - 1):
        if not follows[word]:
            break
        word = follows[word].most_common(1)[0][0]
        out.append(word)
    return " ".join(out)

print(generate("the"))

## 5. Try it yourself

- Add sentences to `corpus`, such as "the dog ran home", and rerun. How do the probabilities change?
- Add a word to the map with your own coordinates and test a new analogy.